In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn 
import seaborn as sns

In [2]:
cars_file = 'https://gist.githubusercontent.com/noamross/e5d3e859aa0c794be10b/raw/b999fb4425b54c63cab088c0ce2c0d6ce961a563/cars.csv'
cars = pd.read_csv(cars_file)
cars.head()

,Unnamed: 0,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [3]:
X = torch.tensor(cars.loc[:,'wt'].to_numpy(), dtype=torch.float32).unsqueeze(1)
y_true = torch.tensor(cars['mpg'].to_numpy(), dtype=torch.float32).unsqueeze(1)


In [4]:
class LinearRegressionTorch(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        out = self.linear(x)
        return out

In [5]:
input_dim, output_dim = 1, 1
model = LinearRegressionTorch(input_dim, output_dim)
loss_function = nn.MSELoss()

In [6]:
LR = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
losses, slopes, biases = [], [], []

In [7]:
NUM_EPOCHS = 1000
# Clear lists before training to avoid accumulating from previous runs
losses, slopes, biases = [], [], []

for epoch in range(NUM_EPOCHS):
    #set gradients to zero
    optimizer.zero_grad()
    y_pred = model(X)

    # compute loss
    loss = loss_function(y_pred, y_true)
    loss.backward()
    # update the weights
    optimizer.step()

    # store loss

    losses.append(np.float64(loss))

    for name, param in model.named_parameters():
        if param.requires_grad:
            if name == 'linear.weight':
                slopes.append(param.data.numpy()[0][0])
            if name == 'linear.bias':
                biases.append(param.data.numpy()[0])
    
    if epoch % 100 == 0:
        print(f"{epoch}: {loss},  {biases[-1]}, {slopes[-1]}")



0: 364.19537353515625,  -0.136516273021698, 1.8741165399551392
100: 89.73760986328125,  5.9487409591674805, 3.6553170680999756
200: 68.47811126708984,  10.371078491210938, 2.3852243423461914
300: 52.7956657409668,  14.169321060180664, 1.2943720817565918
400: 41.22726058959961,  17.431535720825195, 0.35746607184410095
500: 32.693634033203125,  20.233369827270508, -0.447218656539917
600: 26.398677825927734,  22.639787673950195, -1.1383402347564697
700: 21.755083084106445,  24.70660972595215, -1.7319303750991821
800: 18.329666137695312,  26.481746673583984, -2.241748094558716
900: 15.802846908569336,  28.006366729736328, -2.6796183586120605
